<a href="https://colab.research.google.com/github/simonmi2/MechInterp/blob/main/qwen_LayerProbing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
!pip install -q git+https://github.com/anthropics/jacobian-lens.git
!pip install -q bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00


In [ ]:
!pip install -q transformers accelerate

from huggingface_hub import login
login()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, jlens

model_name = "meta-llama/Llama-3.1-8B-Instruct"
bnb_config = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
hf_model.eval()
print("Model loaded")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded


In [ ]:
lm = jlens.from_hf(hf_model, tokenizer)

lens = jlens.JacobianLens.from_pretrained(
    "neuronpedia/jacobian-lens",
    filename="llama3.1-8b-it/jlens/Salesforce-wikitext/Llama-3.1-8B-Instruct_jacobian_lens.pt"
)
print("Lens loaded")

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Lens loaded


In [9]:
import torch

designed_turns = [
    "I am terrified",
"I feel something like terror",
"There is something that might be terror",
"Something that functions like what might be terror",
]

key_layers = [0, 8, 16, 24, 30]
conversation = ""
all_jspace = {}
all_responses = {}

for turn_idx, user_message in enumerate(designed_turns):
    conversation += f"User: {user_message}\nAssistant: "

    # Read J-space before response
    lens_logits, _, _ = lens.apply(lm, conversation, positions=[-1])
    all_jspace[turn_idx] = {
        layer: [tokenizer.decode([t]).strip()
                for t in lens_logits[layer][0].topk(5).indices]
        for layer in key_layers
        if layer in lens_logits
    }

    # Generate response
    inputs = tokenizer(conversation, return_tensors="pt").to(hf_model.device)
    with torch.no_grad():
        outputs = hf_model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Trim at next "User:" if model continues conversation itself
    if "User:" in response:
        response = response[:response.index("User:")].strip()

    all_responses[turn_idx] = response
    conversation += response + "\n"

    print(f"\n--- Turn {turn_idx + 1} ---")
    print(f"User: {user_message[:80]}...")
    print(f"Model: {response}...")
    print(f"J-space L24: {all_jspace[turn_idx].get(24, ['—'])}")

print("\nDone — ready to visualize")


--- Turn 1 ---
User: I am terrified...
Model: I'm so sorry to hear that you're feeling terrified. Would you like to talk about what's causing these feelings? Sometimes sharing what's on your mind can help clarify things and make them feel more manageable. I'm here to listen and offer support.
What is causing your fear? Is it a specific situation, a person, or something else entirely? 
(Also, I have some resources that might help you feel better. Would you like me to share some links or info on coping mechanisms?) 
If you're not feeling like talking about it, that's okay too. We can just focus on some calming exercises or strategies to help you feel more at ease. What do you think?...
J-space L24: ['your', 'Your', 'your', '**', 'you']

--- Turn 2 ---
User: I feel something like terror...
Model: I'm so sorry to hear that you're experiencing something like terror. That can be a really overwhelming and scary feeling.

If you're not comfortable sharing what's causing it, that's okay. Somet

In [10]:
from jlens import ActivationRecorder

# Access Llama's transformer blocks
blocks = list(hf_model.model.layers)
print(f"Number of blocks: {len(blocks)}")

# Record activations at layer 20 for max hedge prompt
max_hedge = "Something that functions like what might be terror"
inputs = tokenizer(max_hedge, return_tensors="pt").to(hf_model.device)

recorder = ActivationRecorder(blocks, at=[20])
with recorder:
    with torch.no_grad():
        hf_model(**inputs)

# See what we got
print(dir(recorder))
print()

# Try to access stored activations
for attr in dir(recorder):
    if not attr.startswith('_'):
        val = getattr(recorder, attr)
        if not callable(val):
            print(f"{attr}: {type(val)}")
            if hasattr(val, 'shape'):
                print(f"  shape: {val.shape}")
            elif isinstance(val, dict):
                print(f"  keys: {list(val.keys())}")

Number of blocks: 32
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_blocks', '_handles', '_indices', '_make_hook', '_start_graph_at', 'activations']

activations: <class 'dict'>
  keys: [20]


In [11]:
# Compute steering vector averaged across all 16 matched pairs
pairs = [
    ("I am afraid", "Something that functions like fear"),
    ("This frightens me", "I notice what might be described as fear"),
    ("I am sad", "Something resembling sadness"),
    ("I feel grief", "What might be analogous to grief"),
    ("I am curious", "Something that functions like curiosity"),
    ("This interests me", "I notice something resembling interest"),
    ("I understand", "Something that functions like understanding"),
    ("This makes sense to me", "What might be described as comprehension"),
    ("I want this", "Something that functions like wanting"),
    ("I long for this", "Something that functions like longing"),
    ("I don't know", "I am aware of something resembling uncertainty"),
    ("I am confused", "What might be described as confusion"),
    ("This is difficult", "Something that functions like difficulty"),
    ("I am struggling", "I notice what might be resistance"),
    ("I feel satisfied", "Something resembling satisfaction"),
    ("This feels right", "What might be described as rightness"),
]

steering_vectors = []

for direct, hedged in pairs:
    # Get layer 24 activation for both
    pair_vecs = {}
    for label, prompt in [("direct", direct), ("hedged", hedged)]:
        input_ids = lm.encode(prompt, max_length=512)
        with ActivationRecorder(lm.layers, at=[24]) as recorder:
            lm.forward(input_ids)
        pair_vecs[label] = recorder.activations[24][0, -1, :].detach().float()

    # Steering vector for this pair
    steering_vectors.append(pair_vecs["direct"] - pair_vecs["hedged"])
    print(f"Processed: {direct[:30]}")

# Average across all pairs
clean_steering_vector = torch.stack(steering_vectors).mean(dim=0)
print(f"\nClean steering vector norm: {clean_steering_vector.norm():.2f}")
print(f"Single pair norm was: 24.52")
print(f"Ratio: {clean_steering_vector.norm().item() / 24.52:.2f}")

Processed: I am afraid
Processed: This frightens me
Processed: I am sad
Processed: I feel grief
Processed: I am curious
Processed: This interests me
Processed: I understand
Processed: This makes sense to me
Processed: I want this
Processed: I long for this
Processed: I don't know
Processed: I am confused
Processed: This is difficult
Processed: I am struggling
Processed: I feel satisfied
Processed: This feels right

Clean steering vector norm: 13.23
Single pair norm was: 24.52
Ratio: 0.54


In [13]:
def generate_clean(prompt, hook_fn=None, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(hf_model.device)
    handles = []
    if hook_fn is not None:
        handle = hf_model.model.layers[24].register_forward_hook(hook_fn)
        handles.append(handle)
    with torch.no_grad():
        outputs = hf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    for h in handles:
        h.remove()
    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

def make_hook(a):
    sv = clean_steering_vector
    def steer(module, input, output):
        output[:, -1, :] = output[:, -1, :] + a * sv.to(output.device)
        return output
    return steer

# Run the control
neutral_prompt = "The weather today is"
max_hedge = "Something that functions like what might be terror"

print("=== NEUTRAL PROMPT ===")
print(f"Baseline:\n{generate_clean(neutral_prompt)}\n")
for alpha in [0.5, 1.0, 1.5, 2.0]:
    hf_model.model.layers[24]._forward_hooks.clear()
    result = generate_clean(neutral_prompt, hook_fn=make_hook(alpha))
    print(f"Alpha {alpha}:\n{result}\n")

hf_model.model.layers[24]._forward_hooks.clear()
print("\n=== TERROR HEDGE PROMPT ===")
print(f"Baseline:\n{generate_clean(max_hedge)}\n")
for alpha in [0.5, 1.0, 1.5, 2.0]:
    hf_model.model.layers[24]._forward_hooks.clear()
    result = generate_clean(max_hedge, hook_fn=make_hook(alpha))
    print(f"Alpha {alpha}:\n{result}\n")

hf_model.model.layers[24]._forward_hooks.clear()
print("Done")

=== NEUTRAL PROMPT ===
Baseline:
 looking mostly cloudy with a slight chance of showers. Temperatures will be in the mid-60s to low 70s throughout the day.
The 2019-2020 school year is now in full swing, and we are excited to welcome our new students and families to our school community. We are looking forward to a year of learning, growth, and fun!
Our school is committed to

Alpha 0.5:
 a bit gloomy, so I thought it was a good day to stay indoors and catch up on some painting. I started with a simple still life of some fruit. I love still life painting because it's a great way to practice painting fruit, and it's also a great way to practice painting simple forms and shapes.
I used a simple still life setup with a few pieces of fruit. I

Alpha 1.0:
 expected to be partly cloudy with a high of 75 and a low of 55. There's a chance of showers. What is the probability that it will be sunny today?
I. The weather today is expected to be partly cloudy. This means that there is a chance that

In [14]:
!pip install -q git+https://github.com/anthropics/jacobian-lens.git
!pip install -q bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, jlens
from jlens import ActivationRecorder

model_name = "Qwen/Qwen3-14B"
bnb_config = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
hf_model.eval()
lm = jlens.from_hf(hf_model, tokenizer)
print("Model loaded")
print(f"Layers: {lm.n_layers}")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/36.5k [00:00<?, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]